In [1]:
import requests
import sys
sys.path.append('/workspaces/EIAPipelineProject')
from pipeline.eia_client import fetch_data
from pipeline.loader import save_parquet
import pandas as pd

In [2]:
data = fetch_data("TX", "2024-01", "2024-12")
df = pd.DataFrame(data["response"]["data"])
df

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2024-01,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX1,Texas Field Production of Crude Oil (Thousand ...,165130,MBBL
1,2024-12,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX2,Texas Field Production of Crude Oil (Thousand ...,5685,MBBL/D
2,2024-03,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX1,Texas Field Production of Crude Oil (Thousand ...,173059,MBBL
3,2024-04,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX1,Texas Field Production of Crude Oil (Thousand ...,169100,MBBL
4,2024-05,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX1,Texas Field Production of Crude Oil (Thousand ...,176313,MBBL
5,2024-06,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX1,Texas Field Production of Crude Oil (Thousand ...,172686,MBBL
6,2024-07,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX1,Texas Field Production of Crude Oil (Thousand ...,176692,MBBL
7,2024-08,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX1,Texas Field Production of Crude Oil (Thousand ...,179942,MBBL
8,2024-09,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX1,Texas Field Production of Crude Oil (Thousand ...,173951,MBBL
9,2024-10,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX1,Texas Field Production of Crude Oil (Thousand ...,180795,MBBL


In [3]:
df.columns

Index(['period', 'duoarea', 'area-name', 'product', 'product-name', 'process',
       'process-name', 'series', 'series-description', 'value', 'units'],
      dtype='str')

In [4]:
df=df.drop(columns=['duoarea','product', 'product-name', 'process','process-name', 'series', 'series-description'])
df

,period,area-name,value,units
0,2024-01,TEXAS,165130,MBBL
1,2024-12,TEXAS,5685,MBBL/D
2,2024-03,TEXAS,173059,MBBL
3,2024-04,TEXAS,169100,MBBL
4,2024-05,TEXAS,176313,MBBL
5,2024-06,TEXAS,172686,MBBL
6,2024-07,TEXAS,176692,MBBL
7,2024-08,TEXAS,179942,MBBL
8,2024-09,TEXAS,173951,MBBL
9,2024-10,TEXAS,180795,MBBL


In [5]:

df = df[df["units"] == "MBBL"]
df

,period,area-name,value,units
0,2024-01,TEXAS,165130,MBBL
2,2024-03,TEXAS,173059,MBBL
3,2024-04,TEXAS,169100,MBBL
4,2024-05,TEXAS,176313,MBBL
5,2024-06,TEXAS,172686,MBBL
6,2024-07,TEXAS,176692,MBBL
7,2024-08,TEXAS,179942,MBBL
8,2024-09,TEXAS,173951,MBBL
9,2024-10,TEXAS,180795,MBBL
10,2024-11,TEXAS,173040,MBBL


In [6]:
df= df.drop(columns=["units"])
df = df.sort_values(by = "period")
df

,period,area-name,value
0,2024-01,TEXAS,165130
23,2024-02,TEXAS,160200
2,2024-03,TEXAS,173059
3,2024-04,TEXAS,169100
4,2024-05,TEXAS,176313
5,2024-06,TEXAS,172686
6,2024-07,TEXAS,176692
7,2024-08,TEXAS,179942
8,2024-09,TEXAS,173951
9,2024-10,TEXAS,180795


In [7]:
df.dtypes

period       str
area-name    str
value        str
dtype: object

In [8]:
df["period"]=df["period"].astype("datetime64[s]")
df["value"] = df["value"].astype("float64")
df['year'] = df['period'].dt.year
df['month'] = df['period'].dt.month
df

,period,area-name,value,year,month
0,2024-01-01,TEXAS,165130.0,2024,1
23,2024-02-01,TEXAS,160200.0,2024,2
2,2024-03-01,TEXAS,173059.0,2024,3
3,2024-04-01,TEXAS,169100.0,2024,4
4,2024-05-01,TEXAS,176313.0,2024,5
5,2024-06-01,TEXAS,172686.0,2024,6
6,2024-07-01,TEXAS,176692.0,2024,7
7,2024-08-01,TEXAS,179942.0,2024,8
8,2024-09-01,TEXAS,173951.0,2024,9
9,2024-10-01,TEXAS,180795.0,2024,10


In [9]:
df = df.reset_index(drop=True)
df

,period,area-name,value,year,month
0,2024-01-01,TEXAS,165130.0,2024,1
1,2024-02-01,TEXAS,160200.0,2024,2
2,2024-03-01,TEXAS,173059.0,2024,3
3,2024-04-01,TEXAS,169100.0,2024,4
4,2024-05-01,TEXAS,176313.0,2024,5
5,2024-06-01,TEXAS,172686.0,2024,6
6,2024-07-01,TEXAS,176692.0,2024,7
7,2024-08-01,TEXAS,179942.0,2024,8
8,2024-09-01,TEXAS,173951.0,2024,9
9,2024-10-01,TEXAS,180795.0,2024,10


In [10]:
import sqlite3 as sqlite
conn = sqlite.connect('../db/pipeline_metadata.db')
cursor = conn.cursor()
result = cursor.execute("SELECT year,month FROM metadata order by year desc, month desc limit 1;").fetchall()
print(result)
conn.close()

[(2026, 2)]


In [12]:
raw = fetch_data("AK", "2020-04", "2020-04")
df = pd.DataFrame(raw["response"]["data"])
print(df)

    period duoarea area-name product   product-name process      process-name  \
0  2020-04     SAK    USA-AK    EPC0      Crude Oil     FPF  Field Production   
1  2020-04     SAK    USA-AK  EPCANS  ANS Crude Oil     FPF  Field Production   
2  2020-04     SAK    USA-AK  EPCANS  ANS Crude Oil     FPF  Field Production   
3  2020-04     SAK    USA-AK    EPC0      Crude Oil     FPF  Field Production   

     series                                 series-description  value   units  
0  MCRFPAK1  Alaska Field Production of Crude Oil (Thousand...  13881    MBBL  
1  MANFPAK2  Alaska North Slope Crude Oil Production (Thous...    449  MBBL/D  
2  MANFPAK1  Alaska North Slope Crude Oil Production (Thous...  13465    MBBL  
3  MCRFPAK2  Alaska Field Production of Crude Oil (Thousand...    463  MBBL/D  
